# Seven-method benchmark: anchored theory and GeoFusion geotechnical-feature ablation

This notebook extends the attached six-method benchmark. It scores seven
methods on exactly the same held-out borehole--endpoint pairs for each rolling
or recursive k-month horizon (k = 1, 3, 6, 12), with 100 runs per horizon.

- T: unanchored multilayer theoretical geotechnical settlement, relative to
  1 January 2017. InSAR observations are never forecast inputs or anchors.
- TA: the SAME layered equation and ratio draw, but anchored to the observed
  settlement at each rolling origin. In recursive evaluation it is anchored
  once to observed December 2020 settlement, then extended by its own
  theoretical increments; no 2021 observations enter recursive forecasts.
  T versus TA tests the influence of observed-origin anchoring, not an extra
  learned geotechnical feature. They have different input information.
- G: geotechnical-only MLP using borehole descriptors and future calendar
  variables, with no observed-settlement inputs.
- A: InSAR-only LSTM; B: GeoFusion using mean borehole features and InSAR.
- B0: identical GeoFusion architecture, transferred temporal branch, tuning,
  random seed and training routine as B. Its STATIC feature array is set to
  zero for fit, validation and test predictions. A and B0 are both needed:
  B vs A mixes architecture and data; B vs B0 isolates the contribution of
  informative static inputs within the fusion architecture.
- P: persistence.

Both theoretical methods use the same global C-alpha/Cc ratio in each run.
T, TA, G, A, B, B0 and P are evaluated on identical eligible held-out
borehole--target pairs for each protocol and horizon.

Training uses pre-2021 data only. The fit/validation borehole split, feature
preprocessing, k-month input and output lengths, and tuned hyperparameters
remain fixed across 100 training seeds. GeoFusion B and B0 are retrained
from the SAME seed and the SAME fitted A temporal branch each run. T has 100
global Uniform(0.03, 0.05) ratio draws instead; P is deterministic. These
sources of variation must not be interpreted as interchangeable.

Rolling scores every 2021 monthly target from its observed k-month sequence.
Recursive starts in December 2020 and advances in k-month forecast blocks;
only block endpoints are scored. Predictions of B and B0 feed back into
THEIR OWN next blocks. Complete monthly windows only; missing months are
never interpolated. All results use RMSE in millimeters. The extra paired
analysis reports run-level RMSE differences and a borehole-level bootstrap
sensitivity interval for B against B0, A and P. The latter is exploratory:
Up to 17 fixed, potentially spatially dependent test boreholes are not a random
sample of future sites.

Update the borehole and InSAR CSV paths in Settings, then run top to bottom.
The notebook does not embed results or assert that GeoFusion must improve.


In [ ]:
# ==============================================================
# IMPORTS
# ============================================================== 

import copy
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

try:
    import optuna
except ImportError as exc:
    raise ImportError(
        "Optuna is required. Install it with: pip install optuna"
    ) from exc

from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import RobustScaler, StandardScaler
from torch.utils.data import Dataset, DataLoader

In [ ]:
# ==============================================================
# USER SETTINGS
# ==============================================================

bh_csv = "../../BM2_Local_BID_11_minus_Table1_XYZ_merged.csv"

insar_ts_csv = (
    "../../../Japan_Airport/script/"
    "vertical_displacement_csv_NewJuly2/"
    "settlement_selected_POIs_positive_downward.csv"
)

target_col = "settlement_mm_positive_downward"

train_start = pd.Timestamp("2017-01-01")
train_end = pd.Timestamp("2020-12-31")
forecast_start = pd.Timestamp("2021-01-01")
forecast_end = pd.Timestamp("2021-12-01")

# Lookback is fixed at L_k = k, not tuned by Optuna.
sequence_length_choices = [1, 3, 6, 12]  # Fixed: L_k = k
max_sequence_length = max(sequence_length_choices)

batch_size = 64
max_epochs = 300
learning_rate = 1.0e-3
weight_decay = 1.0e-4
gradient_clip_norm = 1.0
lstm_hidden = 64
lstm_num_layers = 2
static_hidden = 16
fusion_hidden = 32
dropout = 0.10

validation_fraction = 0.20
early_stopping_patience = 50
early_stopping_min_delta = 1.0e-5

n_simulations = 100
n_borehole_bootstraps = 2000  # Resample held-out boreholes, not individual months.

# Optuna settings. Increase n_trials for a more exhaustive search.
optuna_n_trials_model_a = 25
optuna_n_trials_geofusion = 20
optuna_max_epochs = 150
optuna_patience = 20
optuna_seed = 2026
optuna_n_jobs = 1  # Keep 1 for deterministic GPU studies.

base_random_state = 42
positive_threshold = -10.0

forecast_horizons = [1, 3, 6, 12]

geotechnical_variables = [
    "Depth", "Su", "E50", "Pc", "Cc", "Cv", "PI",
    "LL", "PL", "w", "gamma_t", "Sr", "void_ratio",
]

include_spatial_features = False
spatial_variables = ["x", "y", "z"]

requested_train_bhs = [
    "605", "569", "546", "1148", "578", "1048", "559", "487",
    "596", "549", "568", "437", "545", "579", "634", "1050",
    "995", "567", "4590", "607", "474", "554", "408", "572",
    "1020", "440", "491", "1053", "597", "1049", "489", "893",
    "598", "562", "558", "967", "433", "476", "548", "475",
    "560", "4652", "553", "961", "674", "570", "253", "4804",
    "960", "574", "581", "965", "490", "479", "1008", "564",
    "4625", "1017", "464", "958", "885", "4624", "576", "632",
    "427", "563",
]

requested_test_bhs = [
    "4799", "1006", "4626", "4803", "441", "478",
    "180", "566", "1047", "254", "561", "4806",
    "6133", "488", "630", "555", "886",
]

output_dir = Path("seven_method_geofusion_ablation_100_runs_October12")
output_dir.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
# Constants from the attached theoretical secondary-consolidation notebook.
construction_completion_date = pd.Timestamp("2000-01-01")
settlement_reference_date = pd.Timestamp("2017-01-01")
primary_duration_years = 1.0
ca_ratio_bounds = (0.03, 0.05)
central_ca_ratio = 0.04

# Physics-informed features from the attached geotechnical MLP notebook.
gamma_w = 9.81
cr_ratio_bounds = (0.03, 0.25)
geo_ca_ratio_bounds = (0.01, 0.12)
proxy_quantile_low, proxy_quantile_high = 0.05, 0.95
cv_unit, drainage_condition = "cm2/day", "double"
max_selected_static_features = 24
optuna_n_trials_geotech = 20


In [ ]:
# ==============================================================
# REPRODUCIBILITY + METRICS
# ==============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def overall_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return {"RMSE_mm": float(rmse), "N": len(y_true)}

In [ ]:
# ==============================================================
# PREPROCESSING HELPERS
# ==============================================================

def make_monthly_insar(df):
    records = []
    for poi_id, group in df.groupby("poi_id"):
        group = group.sort_values("date").set_index("date")
        monthly = group[[target_col]].resample("MS").mean()
        monthly = monthly.dropna(subset=[target_col])
        monthly["poi_id"] = str(poi_id)
        records.append(monthly.reset_index())
    return pd.concat(records, ignore_index=True)


def make_mean_geotechnical_features(bh):
    bh = bh.copy()
    bh["borehole_id"] = bh["borehole_id"].astype(str).str.strip()

    variables = list(geotechnical_variables)
    if include_spatial_features:
        variables += spatial_variables
    variables = [c for c in variables if c in bh.columns]

    for col in variables:
        bh[col] = pd.to_numeric(bh[col], errors="coerce")

    mean_features = (
        bh.groupby("borehole_id", as_index=False)[variables]
        .mean()
        .rename(columns={c: f"{c}_mean" for c in variables})
    )

    usable = [
        c for c in mean_features.columns
        if c == "borehole_id" or mean_features[c].notna().any()
    ]
    return mean_features[usable]


def prepare_layers(boreholes):
    boreholes = boreholes.copy()
    boreholes.columns = boreholes.columns.str.strip()
    required = {"borehole_id", "Depth", "e", "Cc"}
    missing = required - set(boreholes.columns)
    if missing:
        raise ValueError(f"Missing borehole columns: {sorted(missing)}")
    boreholes["borehole_id"] = boreholes["borehole_id"].astype(str).str.strip()
    for column in ["Depth", "e", "Cc"]:
        boreholes[column] = pd.to_numeric(boreholes[column], errors="coerce")
    prepared = []
    for borehole_id, group in boreholes.groupby("borehole_id"):
        group = group.dropna(subset=["Depth", "e", "Cc"]).sort_values("Depth").copy()
        if len(group) < 2:
            continue
        depth = group["Depth"].to_numpy(float)
        internal = 0.5 * (depth[:-1] + depth[1:])
        top = max(0.0, depth[0] - 0.5 * (depth[1] - depth[0]))
        bottom = depth[-1] + 0.5 * (depth[-1] - depth[-2])
        group["H"] = np.diff(np.r_[top, internal, bottom])
        group = group[(group["H"] > 0) & (group["e"] > -0.99) & (group["Cc"] >= 0)]
        group["borehole_id"] = str(borehole_id)
        prepared.append(group[["borehole_id", "Depth", "H", "e", "Cc"]])
    if not prepared:
        raise ValueError("No valid multilayer boreholes were available.")
    return pd.concat(prepared, ignore_index=True)

def absolute_secondary_settlement_mm(layers, date, ca_ratio):
    elapsed_years = max((pd.Timestamp(date) - construction_completion_date).days / 365.2425, 0.0)
    time_factor = np.log10((elapsed_years + primary_duration_years) / primary_duration_years)
    ca = layers["Cc"].to_numpy(float) * float(ca_ratio)
    settlement_m = np.sum(layers["H"].to_numpy(float) * ca
                          / (1.0 + layers["e"].to_numpy(float)) * time_factor)
    return float(settlement_m * 1000.0)



def relative_secondary_settlement_mm(layers, date, ratio):
    return absolute_secondary_settlement_mm(layers, date, ratio) - absolute_secondary_settlement_mm(layers, settlement_reference_date, ratio)


def safe_percentile(values, q):
    values = pd.to_numeric(values, errors="coerce").dropna().to_numpy(float)
    if len(values) == 0:
        return np.nan
    return float(np.percentile(values, q))


def weighted_mean(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if not np.any(mask):
        return np.nan
    return float(np.average(values[mask], weights=weights[mask]))


def weighted_sum(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    mask = np.isfinite(values) & np.isfinite(weights)
    if not np.any(mask):
        return np.nan
    return float(np.sum(values[mask] * weights[mask]))


def convert_cv_to_m2_day(cv_values):
    cv_values = pd.to_numeric(cv_values, errors="coerce").astype(float)

    if cv_unit == "cm2/day":
        converted = cv_values * 1.0e-4
    elif cv_unit == "m2/year":
        converted = cv_values / 365.25
    elif cv_unit == "m2/day":
        converted = cv_values
    else:
        raise ValueError("Unsupported cv_unit")

    return converted.clip(lower=1.0e-10)


def robust_unit_interval(series, low_q=0.05, high_q=0.95, neutral=0.5):
    """Robustly map a numeric series to the interval [0, 1]."""
    values = pd.to_numeric(series, errors="coerce").astype(float)
    finite = values[np.isfinite(values)]
    if len(finite) < 2:
        return pd.Series(neutral, index=values.index, dtype=float)
    lower = float(finite.quantile(low_q))
    upper = float(finite.quantile(high_q))
    if not np.isfinite(lower) or not np.isfinite(upper) or upper <= lower:
        return pd.Series(neutral, index=values.index, dtype=float)
    return ((values - lower) / (upper - lower)).clip(0.0, 1.0).fillna(neutral)


def estimate_layerwise_cr_ca(group):
    """Create layer-specific Cr and C_alpha proxy estimates.

    These are data-adaptive engineering proxies because the input table does
    not include unloading-reloading oedometer slopes or secondary creep slopes.
    """
    result = group.copy()
    pi_index = robust_unit_interval(result["PI_calculated"], proxy_quantile_low, proxy_quantile_high)
    ll_index = robust_unit_interval(result["LL"], proxy_quantile_low, proxy_quantile_high)
    e_index = robust_unit_interval(result["e"], proxy_quantile_low, proxy_quantile_high)
    li_index = robust_unit_interval(result["LI"], proxy_quantile_low, proxy_quantile_high)
    ocr_values = pd.to_numeric(result["OCR"], errors="coerce").clip(lower=0.05)
    ocr_index = robust_unit_interval(np.log1p(ocr_values), proxy_quantile_low, proxy_quantile_high)

    cr_over_cc = (
        0.035 + 0.085 * pi_index + 0.035 * ll_index + 0.045 * ocr_index
    ).clip(*cr_ratio_bounds)

    ca_over_cc = (
        0.010 + 0.030 * pi_index + 0.020 * ll_index
        + 0.025 * e_index + 0.025 * li_index
        + 0.020 * (1.0 - ocr_index)
    ).clip(*geo_ca_ratio_bounds)

    result["Cr_over_Cc_estimated"] = cr_over_cc
    result["Ca_over_Cc_estimated"] = ca_over_cc
    result["Cr"] = result["Cc"] * cr_over_cc
    result["Ca"] = result["Cc"] * ca_over_cc
    result["Cr_ratio"] = result["Cr"] / (1.0 + result["e"])
    result["Ca_ratio"] = result["Ca"] / (1.0 + result["e"])
    return result


def prepare_physics_informed_borehole_features(bh_df):
    bh = bh_df.copy()
    bh.columns = bh.columns.str.strip()

    required = {
        "borehole_id", "Depth", "gamma_t", "e", "Su", "E50",
        "Pc", "Cc", "Cv", "LL", "PL", "w", "Sr"
    }
    missing = required - set(bh.columns)
    if missing:
        raise ValueError(
            "Missing required borehole columns: "
            + ", ".join(sorted(missing))
        )

    bh["borehole_id"] = bh["borehole_id"].astype(str).str.strip()

    numeric_columns = [
        "Depth", "Sr", "gamma_t", "e", "LL", "PL", "w",
        "Su", "E50", "Pc", "Cc", "Cv", "PI",
        "e_squared", "moisture_void_ratio", "x", "y", "elevation"
    ]

    for col in numeric_columns:
        if col in bh.columns:
            bh[col] = pd.to_numeric(bh[col], errors="coerce")

    rows = []

    for borehole_id, group in bh.groupby("borehole_id"):
        group = group.sort_values("Depth").copy()
        group = group.dropna(subset=["Depth"])

        if len(group) < 2:
            continue

        z = group["Depth"].to_numpy(float)
        z_mid = 0.5 * (z[:-1] + z[1:])
        z_top = max(0.0, z[0] - 0.5 * (z[1] - z[0]))
        z_bottom = z[-1] + 0.5 * (z[-1] - z[-2])
        bounds = np.concatenate([[z_top], z_mid, [z_bottom]])
        H = np.diff(bounds)
        H = np.clip(H, 1.0e-6, None)
        group["H"] = H

        group["PI_calculated"] = group["LL"] - group["PL"]
        group["LI"] = (
            (group["w"] - group["PL"])
            / group["PI_calculated"].replace(0.0, np.nan)
        )
        group["Cc_ratio"] = group["Cc"] / (1.0 + group["e"])
        group["gamma_sub"] = group["gamma_t"] - gamma_w

        # Effective vertical stress at layer centers by cumulative
        # submerged unit weight, consistent with the notebook concept.
        sigma_centers = []
        sigma_at_top = 0.0
        for _, layer in group.iterrows():
            gamma_sub = max(float(layer["gamma_sub"]), 0.1)
            h = float(layer["H"])
            sigma_center = sigma_at_top + gamma_sub * h / 2.0
            sigma_centers.append(max(sigma_center, 1.0))
            sigma_at_top += gamma_sub * h

        group["sigma_v0_eff"] = sigma_centers
        group["OCR"] = group["Pc"] / group["sigma_v0_eff"]
        group["Su_ratio"] = group["Su"] / group["sigma_v0_eff"]
        group["E50_ratio"] = group["E50"] / group["sigma_v0_eff"]

        # Estimate Cr and C_alpha from the available layer data after OCR
        # has been calculated.
        group = estimate_layerwise_cr_ca(group)

        group["Cv_m2_day"] = convert_cv_to_m2_day(group["Cv"])

        if drainage_condition == "double":
            group["Hdr_m"] = group["H"] / 2.0
        elif drainage_condition == "single":
            group["Hdr_m"] = group["H"]
        else:
            raise ValueError(
                "drainage_condition must be 'single' or 'double'"
            )

        group["Hdr_m"] = group["Hdr_m"].clip(lower=1.0e-6)
        group["characteristic_time_days"] = (
            group["Hdr_m"] ** 2 / group["Cv_m2_day"]
        )

        total_thickness = float(np.sum(group["H"]))
        row = {
            "borehole_id": str(borehole_id),
            "n_layers": len(group),
            "profile_top_depth": float(group["Depth"].min()),
            "profile_bottom_depth": float(group["Depth"].max()),
            "profile_depth_range": float(
                group["Depth"].max() - group["Depth"].min()
            ),
            "total_layer_thickness": total_thickness,
        }

        summary_variables = [
            "Sr", "gamma_t", "gamma_sub", "e", "LL", "PL", "w",
            "PI_calculated", "LI", "Su", "E50", "Pc", "Cc", "Cv",
            "Cc_ratio", "Cr", "Ca", "Cr_over_Cc_estimated",
            "Ca_over_Cc_estimated", "Cr_ratio", "Ca_ratio",
            "sigma_v0_eff", "OCR",
            "Su_ratio", "E50_ratio", "Cv_m2_day", "Hdr_m",
            "characteristic_time_days"
        ]

        for col in summary_variables:
            values = pd.to_numeric(group[col], errors="coerce")
            row[f"{col}_weighted_mean"] = weighted_mean(values, group["H"])
            row[f"{col}_mean"] = values.mean()
            row[f"{col}_median"] = values.median()
            row[f"{col}_std"] = values.std()
            row[f"{col}_min"] = values.min()
            row[f"{col}_max"] = values.max()
            row[f"{col}_p10"] = safe_percentile(values, 10)
            row[f"{col}_p90"] = safe_percentile(values, 90)

        # Integrated compressibility and creep proxies.
        row["integrated_Cc_ratio_H"] = weighted_sum(
            group["Cc_ratio"], group["H"]
        )
        row["integrated_Cr_ratio_H"] = weighted_sum(
            group["Cr_ratio"], group["H"]
        )
        row["integrated_Ca_ratio_H"] = weighted_sum(
            group["Ca_ratio"], group["H"]
        )
        row["low_OCR_thickness_fraction"] = weighted_mean(
            (group["OCR"] <= 1.5).astype(float), group["H"]
        )
        row["low_Su_thickness_fraction"] = weighted_mean(
            (group["Su"] <= group["Su"].median()).astype(float),
            group["H"],
        )
        row["low_E50_thickness_fraction"] = weighted_mean(
            (group["E50"] <= group["E50"].median()).astype(float),
            group["H"],
        )

        if include_spatial_features:
            for col in ["x", "y", "elevation"]:
                if col in group.columns:
                    row[f"{col}_mean"] = group[col].mean()

        rows.append(row)

    features = pd.DataFrame(rows)

    if features.empty:
        raise ValueError("No borehole feature rows could be created.")

    usable_cols = [
        col for col in features.columns
        if col == "borehole_id" or features[col].notna().any()
    ]

    return features[usable_cols]


def calendar_features(months):
    """Target calendar descriptors from the attached geotechnical model."""
    dates = pd.DatetimeIndex(months)
    elapsed = (dates.year - train_start.year) * 12 + dates.month - train_start.month
    elapsed = np.asarray(elapsed, dtype=float)
    days = np.maximum(elapsed, 0) * 30.4375
    secondary = np.zeros(len(days))
    mask = days > 365.25
    secondary[mask] = np.log10(days[mask] / 365.25)
    m = dates.month.to_numpy(dtype=float)
    return np.stack([
        elapsed, elapsed / 12, np.log1p(np.maximum(elapsed, 0)),
        np.sqrt(np.maximum(elapsed, 0)), np.sin(2*np.pi*m/12),
        np.cos(2*np.pi*m/12), np.minimum(days / 365.25, 1),
        secondary,
    ], axis=1)

In [ ]:
# ==============================================================
# DATASETS
# ==============================================================

class InSARDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).reshape(len(y), -1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class FusionDataset(Dataset):
    def __init__(self, X_dynamic, X_static, y):
        self.X_dynamic = torch.tensor(X_dynamic, dtype=torch.float32)
        self.X_static = torch.tensor(X_static, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).reshape(len(y), -1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_dynamic[idx], self.X_static[idx], self.y[idx]

In [ ]:
# ==============================================================
# MODEL A: PARAMETERIZED InSAR-ONLY LSTM
# ============================================================== 

class InSAROnlyLSTM(nn.Module):
    def __init__(self, params):
        super().__init__()
        hidden_size = params["lstm_hidden"]
        num_layers = params["lstm_num_layers"]
        model_dropout = params["dropout"]

        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=model_dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(model_dropout),
            nn.Linear(hidden_size, params["horizon"]),
        )

    def forward(self, x):
        output, _ = self.lstm(x)
        embedding = output[:, -1, :]
        return self.head(embedding)


In [ ]:
# ==============================================================
# MODEL B: TRANSFER-INITIALIZED GEOTECHNICAL LATE FUSION
# ============================================================== 

class GeoFusionLSTM(nn.Module):
    def __init__(self, static_input_size, params):
        super().__init__()
        hidden_size = params["lstm_hidden"]
        num_layers = params["lstm_num_layers"]
        model_dropout = params["dropout"]
        static_size = params["static_hidden"]
        fusion_size = params["fusion_hidden"]

        self.lstm = nn.LSTM(
            input_size=1, hidden_size=hidden_size, num_layers=num_layers,
            dropout=model_dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.insar_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size), nn.ReLU(),
            nn.Dropout(model_dropout), nn.Linear(hidden_size, params["horizon"]),
        )
        self.static_encoder = nn.Sequential(
            nn.Linear(static_input_size, static_size), nn.ReLU(),
            nn.LayerNorm(static_size), nn.Dropout(model_dropout),
            nn.Linear(static_size, static_size), nn.ReLU(),
        )
        self.correction_head = nn.Sequential(
            nn.Linear(hidden_size + static_size, fusion_size), nn.ReLU(),
            nn.Dropout(model_dropout), nn.Linear(fusion_size, params["horizon"]),
        )
        self.correction_scale = nn.Parameter(torch.tensor(0.10))

    def initialize_from_insar(self, insar_model):
        """Transfer the optimized temporal representation from Model A."""
        self.lstm.load_state_dict(copy.deepcopy(insar_model.lstm.state_dict()))
        self.insar_head.load_state_dict(copy.deepcopy(insar_model.head.state_dict()))

    def forward(self, dynamic_x, static_x):
        output, _ = self.lstm(dynamic_x)
        temporal_embedding = output[:, -1, :]
        insar_prediction = self.insar_head(temporal_embedding)
        static_embedding = self.static_encoder(static_x)
        correction = self.correction_head(
            torch.cat([temporal_embedding, static_embedding], dim=1)
        )
        return insar_prediction + self.correction_scale * correction


class GeotechnicalTimeMLP(nn.Module):
    """Static soil features + k known future calendar steps, k outputs."""
    def __init__(self, input_size, params):
        super().__init__()
        hidden = params["geo_hidden"]
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden), nn.ReLU(), nn.LayerNorm(hidden),
            nn.Dropout(params["geo_dropout"]),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, params["horizon"]),
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
# ==============================================================
# LOAD + PREPARE DATA
# ==============================================================

bh = pd.read_csv(bh_csv)
insar = pd.read_csv(insar_ts_csv)

bh.columns = bh.columns.str.strip()
insar.columns = insar.columns.str.strip()

insar["poi_id"] = insar["poi_id"].astype(str).str.strip()
insar["date"] = pd.to_datetime(insar["date"], errors="coerce")
insar[target_col] = pd.to_numeric(insar[target_col], errors="coerce")
insar = insar.dropna(subset=["poi_id", "date", target_col])

positive_mask = insar.loc[insar["date"] <= train_end].groupby("poi_id")[target_col].apply(
    lambda x: (x > positive_threshold).all()
)
valid_positive = set(positive_mask[positive_mask].index.astype(str))
insar = insar[insar["poi_id"].isin(valid_positive)].copy()

monthly = make_monthly_insar(insar)
geo_features = make_mean_geotechnical_features(bh)
layers = prepare_layers(bh)
layer_lookup = {pid: group.copy() for pid, group in layers.groupby("borehole_id")}
geotech_features = prepare_physics_informed_borehole_features(bh)
geotech_features["borehole_id"] = geotech_features["borehole_id"].astype(str)
feature_ids = set(geotech_features["borehole_id"]) & set(layer_lookup)
geo_features["borehole_id"] = geo_features["borehole_id"].astype(str)

data = monthly.merge(
    geo_features,
    left_on="poi_id",
    right_on="borehole_id",
    how="inner",
)

# Keep only boreholes that can support every requested comparison.
required_test_months = pd.date_range(forecast_start, forecast_end, freq="MS")
required_history_months = pd.date_range(
    train_end.to_period("M").to_timestamp() - pd.DateOffset(months=2 * max_sequence_length - 1),
    train_end.to_period("M").to_timestamp(),
    freq="MS",
)

valid_boreholes = []
for poi_id, group in data.groupby("poi_id"):
    dates = set(pd.to_datetime(group["date"]).dt.to_period("M").dt.to_timestamp())
    n_train = ((group["date"] >= train_start) & (group["date"] <= train_end)).sum()
    has_history = all(d in dates for d in required_history_months)
    has_test = all(d in dates for d in required_test_months)
    if n_train > (max_sequence_length + 1) and has_history and has_test:
        valid_boreholes.append(str(poi_id))

valid_boreholes = set(valid_boreholes) & feature_ids
train_bhs = [b for b in requested_train_bhs if b in valid_boreholes]
test_bhs = [b for b in requested_test_bhs if b in valid_boreholes]

if len(train_bhs) < 2:
    raise RuntimeError("Too few valid training boreholes after common-data filtering.")
if len(test_bhs) == 0:
    raise RuntimeError("No valid testing boreholes remain after common-data filtering.")

data = data[data["poi_id"].isin(set(train_bhs) | set(test_bhs))].copy()

print("Common training boreholes:", len(train_bhs))
print("Common testing boreholes :", len(test_bhs))
print("Test boreholes:", test_bhs)

In [ ]:
# ==============================================================
# FIXED FIT / VALIDATION BOREHOLE SPLIT FOR ALL HORIZONS
# ==============================================================

train_bh_df = pd.DataFrame({"poi_id": train_bhs})
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=validation_fraction,
    random_state=base_random_state,
)
fit_idx, val_idx = next(
    splitter.split(train_bh_df, groups=train_bh_df["poi_id"])
)
fit_bhs = train_bh_df.iloc[fit_idx]["poi_id"].tolist()
val_bhs = train_bh_df.iloc[val_idx]["poi_id"].tolist()

print("Fit boreholes       :", len(fit_bhs))
print("Validation boreholes:", len(val_bhs))

In [ ]:
# ==============================================================
# TRAINING-ONLY SCALING
# ==============================================================

static_cols = [c for c in geo_features.columns if c != "borehole_id"]
for col in static_cols:
    data[col] = pd.to_numeric(data[col], errors="coerce")

fit_mask = (
    data["poi_id"].isin(fit_bhs)
    & (data["date"] >= train_start)
    & (data["date"] <= train_end)
)
fit_df = data.loc[fit_mask].copy()

settlement_scaler = StandardScaler()
settlement_scaler.fit(fit_df[[target_col]])

static_fit = fit_df.drop_duplicates("poi_id").set_index("poi_id")
static_imputer = SimpleImputer(strategy="median", add_indicator=True)
static_fit_imputed = static_imputer.fit_transform(static_fit[static_cols])

static_scaler = RobustScaler()
static_scaler.fit(static_fit_imputed)

data_scaled = data.copy()
data_scaled["settlement_scaled"] = settlement_scaler.transform(
    data[[target_col]]
).ravel()

static_bh = (
    data[["poi_id"] + static_cols]
    .drop_duplicates("poi_id")
    .set_index("poi_id")
)
static_scaled = static_scaler.transform(
    static_imputer.transform(static_bh[static_cols])
)
static_lookup = {
    bh_id: static_scaled[i].astype(np.float32)
    for i, bh_id in enumerate(static_bh.index)
}

print("Static feature dimension after imputation:", static_scaled.shape[1])

In [ ]:
# Geotechnical feature ranking and preprocessing: fit boreholes and
# pre-2021 observations only. Validation/test targets never select features.
raw_geo = geotech_features.set_index("borehole_id")
raw_geo = raw_geo.loc[fit_bhs + val_bhs + test_bhs]
usable_geo_cols = [
    c for c in raw_geo.columns if raw_geo.loc[fit_bhs, c].notna().any()
]
geo_imputer = SimpleImputer(strategy="median", add_indicator=True)
fit_imp = geo_imputer.fit_transform(raw_geo.loc[fit_bhs, usable_geo_cols])
all_imp = geo_imputer.transform(raw_geo[usable_geo_cols])
indicator_cols = [
    f"{usable_geo_cols[i]}_missing" for i in geo_imputer.indicator_.features_
]
geo_names = usable_geo_cols + indicator_cols
fit_target_means = data.loc[
    data["poi_id"].isin(fit_bhs) & (data["date"] <= train_end)
].groupby("poi_id")[target_col].mean()
if len(fit_bhs) >= 8:
    mi = mutual_info_regression(
        fit_imp, fit_target_means.loc[fit_bhs].to_numpy(float),
        random_state=base_random_state,
    )
    selected_geo = pd.Series(mi, index=geo_names).nlargest(
        min(max_selected_static_features, len(geo_names))
    ).index.tolist()
else:
    selected_geo = geo_names[:max_selected_static_features]
selected_idx = [geo_names.index(name) for name in selected_geo]
geo_scaler = RobustScaler().fit(fit_imp[:, selected_idx])
geo_transformed = geo_scaler.transform(all_imp[:, selected_idx])
geo_lookup = {
    pid: geo_transformed[i].astype(np.float32)
    for i, pid in enumerate(raw_geo.index)
}
calendar_fit_dates = data.loc[
    data["poi_id"].isin(fit_bhs) & data["date"].between(train_start, train_end),
    "date",
]
calendar_scaler = StandardScaler().fit(calendar_features(calendar_fit_dates))
pd.DataFrame({"selected_geotechnical_feature": selected_geo}).to_csv(
    output_dir / "selected_geotechnical_features.csv", index=False
)
print("Selected geotechnical features:", len(selected_geo))

In [ ]:
# Build k-month input blocks with k future monthly targets. The k-th
# target is the direct k-month-ahead value scored in both protocols.
def future_dates(origin, horizon):
    return pd.date_range(origin + pd.DateOffset(months=1), periods=horizon, freq="MS")


def history_dates(origin, horizon):
    return pd.date_range(
        origin - pd.DateOffset(months=horizon - 1), periods=horizon, freq="MS"
    )


def geotechnical_input(poi_id, origin, horizon):
    known_calendar = calendar_scaler.transform(
        calendar_features(future_dates(origin, horizon))
    )
    return np.r_[geo_lookup[poi_id], known_calendar.ravel()].astype(np.float32)


def build_horizon_dataset(horizon):
    arrays = {}
    for part in ("fit", "val", "test"):
        arrays[part] = {name: [] for name in ("dynamic", "static", "geo", "y", "meta")}
    for poi_id, group in data_scaled.groupby("poi_id"):
        group = group.copy()
        group["month"] = pd.to_datetime(group["date"]).dt.to_period("M").dt.to_timestamp()
        if group["month"].duplicated().any():
            raise ValueError(f"Duplicate monthly observation for {poi_id}")
        group = group.set_index("month").sort_index()
        if poi_id in fit_bhs:
            part = "fit"
        elif poi_id in val_bhs:
            part = "val"
        elif poi_id in test_bhs:
            part = "test"
        else:
            continue
        targets = (
            pd.date_range(train_start, train_end, freq="MS")
            if part != "test"
            else pd.date_range(forecast_start, forecast_end, freq="MS")
        )
        for target in targets:
            origin = target - pd.DateOffset(months=horizon)
            past, future = history_dates(origin, horizon), future_dates(origin, horizon)
            if not all(d in group.index for d in past) or not all(
                d in group.index for d in future
            ):
                continue
            # All fit and validation block targets must be pre-2021.
            if part != "test" and (future[-1] > train_end or past[0] < train_start):
                continue
            observed = group.loc[future, target_col].to_numpy(float)
            history = group.loc[past, "settlement_scaled"].to_numpy(float)
            if not np.isfinite(observed).all() or not np.isfinite(history).all():
                continue
            out = arrays[part]
            out["dynamic"].append(history.reshape(horizon, 1))
            out["static"].append(static_lookup[poi_id])
            out["geo"].append(geotechnical_input(poi_id, origin, horizon))
            out["y"].append(group.loc[future, "settlement_scaled"].to_numpy(float))
            out["meta"].append({
                "poi_id": poi_id, "origin_date": origin,
                "target_date": target, "Horizon_months": horizon,
                "observed_settlement_mm": float(observed[-1]),
            })
    result = {}
    for part, values in arrays.items():
        if not values["meta"]:
            raise ValueError(f"No eligible {part} examples at k={horizon}")
        for key in ("dynamic", "static", "geo", "y"):
            result[f"X_{part}_{key}" if key != "y" else f"y_{part}"] = (
                np.asarray(values[key], dtype=np.float32)
            )
        result[f"meta_{part}"] = pd.DataFrame(values["meta"])
    result["y_test_mm"] = settlement_scaler.inverse_transform(
        result["y_test"].reshape(-1, 1)
    ).reshape(result["y_test"].shape)[:, -1]
    return result


datasets = {h: build_horizon_dataset(h) for h in forecast_horizons}
for h, ds in datasets.items():
    for part in ("fit", "val", "test"):
        if ds[f"X_{part}_dynamic"].shape[1:] != (h, 1):
            raise AssertionError(f"Required L_k=k violated for {part}, k={h}")
        if ds[f"y_{part}"].shape[1] != h:
            raise AssertionError(f"Expected k output months for {part}, k={h}")
    print(f"k={h}: fit={len(ds['y_fit'])}, val={len(ds['y_val'])}, "
          f"sequence_length={ds['X_fit_dynamic'].shape[1]}, "
          f"rolling endpoints={len(ds['y_test_mm'])}")

In [ ]:
# ==============================================================
# TRAINING WITH EARLY STOPPING, PRUNING, AND WEIGHT TRANSFER
# ============================================================== 

def evaluate_loss(model, loader, model_type):
    model.eval(); criterion = nn.MSELoss(); losses = []
    with torch.no_grad():
        for batch in loader:
            if model_type == "insar":
                xd, y = batch; xd, y = xd.to(device), y.to(device)
                prediction = model(xd)
            else:
                xd, xs, y = batch
                xd, xs, y = xd.to(device), xs.to(device), y.to(device)
                prediction = model(xd, xs)
            losses.append(criterion(prediction[:, -1:], y[:, -1:]).item())
    return float(np.mean(losses))


def train_model(ds, seed, params, model_type, pretrained_insar=None,
                trial=None, epoch_limit=None, patience_limit=None,
                ablate_static=False):
    set_seed(seed)
    if model_type == "insar":
        train_set = InSARDataset(ds["X_fit_dynamic"], ds["y_fit"])
        val_set = InSARDataset(ds["X_val_dynamic"], ds["y_val"])
        model = InSAROnlyLSTM(params).to(device)
    else:
        # Same architecture and optimization as B. Only the STATIC values
        # change for B0, consistently at fit, validation and forecast time.
        static_fit = (np.zeros_like(ds["X_fit_static"]) if ablate_static
                      else ds["X_fit_static"])
        static_val = (np.zeros_like(ds["X_val_static"]) if ablate_static
                      else ds["X_val_static"])
        train_set = FusionDataset(ds["X_fit_dynamic"], static_fit, ds["y_fit"])
        val_set = FusionDataset(ds["X_val_dynamic"], static_val, ds["y_val"])
        model = GeoFusionLSTM(ds["X_fit_static"].shape[1], params).to(device)
        if pretrained_insar is None:
            raise ValueError("GeoFusion requires the corresponding fitted Model A.")
        model.initialize_from_insar(pretrained_insar)

    train_loader = DataLoader(train_set, batch_size=params["batch_size"], shuffle=True)
    val_loader = DataLoader(val_set, batch_size=params["batch_size"], shuffle=False)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=params["learning_rate"],
        weight_decay=params["weight_decay"],
    )
    criterion = nn.MSELoss()
    epochs = max_epochs if epoch_limit is None else epoch_limit
    patience = early_stopping_patience if patience_limit is None else patience_limit
    best_loss, best_state, counter = np.inf, None, 0

    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            optimizer.zero_grad()
            if model_type == "insar":
                xd, y = batch; xd, y = xd.to(device), y.to(device)
                loss = criterion(model(xd), y)
            else:
                xd, xs, y = batch
                xd, xs, y = xd.to(device), xs.to(device), y.to(device)
                loss = criterion(model(xd, xs), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), params["gradient_clip_norm"])
            optimizer.step()

        val_loss = evaluate_loss(model, val_loader, model_type)
        if trial is not None:
            trial.report(val_loss, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()
        if val_loss < best_loss - early_stopping_min_delta:
            best_loss, best_state, counter = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            counter += 1
        if counter >= patience:
            break
    if best_state is None:
        best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model, best_loss



def train_geotechnical_model(ds, seed, params, trial=None,
                            epoch_limit=None, patience_limit=None):
    """Soil and future calendar MLP based on the attached geotechnical model."""
    set_seed(seed)
    xfit = torch.tensor(ds["X_fit_geo"], dtype=torch.float32)
    yfit = torch.tensor(ds["y_fit"], dtype=torch.float32)
    xval = torch.tensor(ds["X_val_geo"], dtype=torch.float32, device=device)
    yval = torch.tensor(ds["y_val"], dtype=torch.float32, device=device)
    model = GeotechnicalTimeMLP(xfit.shape[1], params).to(device)
    loader = DataLoader(
        torch.utils.data.TensorDataset(xfit, yfit),
        batch_size=params["batch_size"], shuffle=True,
    )
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=params["learning_rate"],
        weight_decay=params["weight_decay"],
    )
    best_loss, state, counter = np.inf, None, 0
    epochs = max_epochs if epoch_limit is None else epoch_limit
    patience = early_stopping_patience if patience_limit is None else patience_limit
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = nn.functional.mse_loss(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), params["gradient_clip_norm"])
            optimizer.step()
        model.eval()
        with torch.no_grad():
            score = nn.functional.mse_loss(model(xval)[:, -1], yval[:, -1]).item()
        if trial is not None:
            trial.report(score, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()
        if score < best_loss - early_stopping_min_delta:
            best_loss, state, counter = score, copy.deepcopy(model.state_dict()), 0
        else:
            counter += 1
        if counter >= patience:
            break
    if state is not None:
        model.load_state_dict(state)
    return model, best_loss

In [ ]:
# Optuna on pre-2021 boreholes only. L_k = k is fixed, not tuned.
best_params_registry = {"A": {}, "B": {}, "G": {}}
study_registry = {"A": {}, "B": {}, "G": {}}

def new_study(label, seed):
    return optuna.create_study(
        study_name=label, direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=seed),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10),
    )

for horizon in forecast_horizons:
    ds = datasets[horizon]
    def objective_a(trial, h=horizon, train_ds=ds):
        layers = trial.suggest_int("lstm_num_layers", 1, 3)
        params = {
            "horizon": h,
            "lstm_num_layers": layers,
            "lstm_hidden": trial.suggest_categorical("lstm_hidden", [32, 64, 128]),
            "dropout": trial.suggest_float("dropout", 0, 0.4) if layers > 1 else 0.0,
            "learning_rate": trial.suggest_float("learning_rate", 1e-4, 5e-3, log=True),
            "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
            "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128]),
            "gradient_clip_norm": trial.suggest_float("gradient_clip_norm", 0.5, 5, log=True),
        }
        _, loss = train_model(
            train_ds, optuna_seed + h, params, "insar", trial=trial,
            epoch_limit=optuna_max_epochs, patience_limit=optuna_patience,
        )
        return loss

    sa = new_study(f"k{horizon}_InSAR", optuna_seed + horizon)
    sa.optimize(objective_a, n_trials=optuna_n_trials_model_a, n_jobs=1)
    pa = dict(sa.best_params)
    pa.update(horizon=horizon, sequence_length=horizon)
    pa.setdefault("dropout", 0.0)
    reference_a, _ = train_model(ds, optuna_seed + horizon, pa, "insar")

    def objective_b(trial, h=horizon, train_ds=ds, base=pa, ref=reference_a):
        params = dict(base)
        params.update({
            "static_hidden": trial.suggest_categorical("static_hidden", [8, 16, 32, 64]),
            "fusion_hidden": trial.suggest_categorical("fusion_hidden", [16, 32, 64, 128]),
            "learning_rate": trial.suggest_float("learning_rate", 1e-5, 2e-3, log=True),
            "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
            "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128]),
            "gradient_clip_norm": trial.suggest_float("gradient_clip_norm", 0.5, 5, log=True),
        })
        _, loss = train_model(
            train_ds, optuna_seed + 100 + h, params, "fusion", pretrained_insar=ref,
            trial=trial, epoch_limit=optuna_max_epochs, patience_limit=optuna_patience,
        )
        return loss

    sb = new_study(f"k{horizon}_GeoFusion", optuna_seed + 100 + horizon)
    sb.optimize(objective_b, n_trials=optuna_n_trials_geofusion, n_jobs=1)
    pb = dict(pa)
    pb.update(sb.best_params)

    def objective_g(trial, h=horizon, train_ds=ds):
        params = {
            "horizon": h,
            "geo_hidden": trial.suggest_categorical("geo_hidden", [32, 64, 128]),
            "geo_dropout": trial.suggest_float("geo_dropout", 0, 0.4),
            "learning_rate": trial.suggest_float("learning_rate", 1e-4, 5e-3, log=True),
            "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
            "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128]),
            "gradient_clip_norm": trial.suggest_float("gradient_clip_norm", 0.5, 5, log=True),
        }
        _, loss = train_geotechnical_model(
            train_ds, optuna_seed + 200 + h, params, trial=trial,
            epoch_limit=optuna_max_epochs, patience_limit=optuna_patience,
        )
        return loss

    sg = new_study(f"k{horizon}_GeotechMLP", optuna_seed + 200 + horizon)
    sg.optimize(objective_g, n_trials=optuna_n_trials_geotech, n_jobs=1)
    pg = dict(sg.best_params)
    pg["horizon"] = horizon
    if pa["sequence_length"] != horizon:
        raise AssertionError(f"Model settings violate L_k=k at k={horizon}")
    best_params_registry["A"][horizon] = pa
    best_params_registry["B"][horizon] = pb
    best_params_registry["G"][horizon] = pg
    study_registry["A"][horizon] = sa
    study_registry["B"][horizon] = sb
    study_registry["G"][horizon] = sg
    for label, study in (("A", sa), ("B", sb), ("G", sg)):
        study.trials_dataframe().to_csv(
            output_dir / f"optuna_{label}_k{horizon}.csv", index=False
        )

with open(output_dir / "optuna_best_parameters.json", "w") as f:
    json.dump({model: {str(h): p for h, p in config.items()}
               for model, config in best_params_registry.items()}, f, indent=2)

In [ ]:
# Forecast helpers. All outputs are k monthly values; the final value is
# the direct k-month prediction. T uses the unanchored 2017-referenced
# layered law; TA uses the SAME layered law anchored at a forecast origin.
def inverse_block(scaled):
    return settlement_scaler.inverse_transform(
        np.asarray(scaled, dtype=float).reshape(-1, 1)
    ).ravel()


def predict_model_block(key, fitted, sequence, static, geo):
    past = np.asarray(sequence, dtype=np.float32).reshape(1, -1, 1)
    xd = torch.as_tensor(past, dtype=torch.float32, device=device)
    with torch.no_grad():
        if key == "A":
            pred = fitted.eval()(xd)
        elif key in ("B", "B0"):
            static_input = np.zeros_like(static) if key == "B0" else static
            xs = torch.as_tensor(static_input, dtype=torch.float32, device=device).reshape(1, -1)
            pred = fitted.eval()(xd, xs)
        elif key == "G":
            xg = torch.as_tensor(geo, dtype=torch.float32, device=device).reshape(1, -1)
            pred = fitted.eval()(xg)
        else:
            raise ValueError(key)
    return np.asarray(pred.detach().cpu().numpy()).ravel().astype(np.float32)


def theoretical_block(poi_id, origin, horizon, ratio):
    """Geotechnical-only curve, zero at the fixed 2017 reference, never InSAR-anchored."""
    layers_bh = layer_lookup[poi_id]
    return np.array([
        relative_secondary_settlement_mm(layers_bh, d, ratio)
        for d in future_dates(origin, horizon)
    ], dtype=float)


def anchored_theoretical_block(poi_id, origin, origin_mm, horizon, ratio):
    """Observed-origin settlement plus theoretical increments, in mm."""
    layers_bh = layer_lookup[poi_id]
    theoretical_origin = absolute_secondary_settlement_mm(layers_bh, origin, ratio)
    return np.array([
        origin_mm + absolute_secondary_settlement_mm(layers_bh, d, ratio)
        - theoretical_origin
        for d in future_dates(origin, horizon)
    ], dtype=float)


def make_forecast_rows(protocol, horizon, simulation, seed, models, ratio):
    ds = datasets[horizon]
    rows = []
    if protocol == "rolling":
        for i, record in ds["meta_test"].iterrows():
            poi_id = record["poi_id"]
            origin = record["origin_date"]
            dynamic = ds["X_test_dynamic"][i, :, 0]
            origin_mm = float(inverse_block([dynamic[-1]])[0])
            static = ds["X_test_static"][i]
            geo = ds["X_test_geo"][i]
            forecasts = {
                "P": np.full(horizon, origin_mm),
                "T": theoretical_block(poi_id, origin, horizon, ratio),
                "TA": anchored_theoretical_block(
                    poi_id, origin, origin_mm, horizon, ratio
                ),
            }
            for key in ("A", "B", "B0", "G"):
                forecasts[key] = inverse_block(predict_model_block(
                    key, models[key], dynamic, static, geo
                ))
            for key, block in forecasts.items():
                rows.append({
                    "Protocol": protocol, "Horizon_months": horizon,
                    "Simulation": simulation, "Seed": seed if key in ("A", "B", "B0", "G") else np.nan,
                    "Model_key": key, "poi_id": poi_id,
                    "origin_date": origin, "target_date": record["target_date"],
                    "observed_settlement_mm": record["observed_settlement_mm"],
                    "predicted_settlement_mm": float(block[-1]),
                    "C_alpha_over_Cc": ratio if key in ("T", "TA") else np.nan,
                })
    else:
        initial = train_end.to_period("M").to_timestamp()
        for poi_id in test_bhs:
            group = data_scaled.loc[data_scaled["poi_id"] == poi_id].copy()
            group["month"] = pd.to_datetime(group["date"]).dt.to_period("M").dt.to_timestamp()
            group = group.set_index("month")
            months = history_dates(initial, horizon)
            if not all(d in group.index for d in months):
                raise ValueError(f"Missing December 2020 history for {poi_id}")
            initial_scaled = group.loc[months, "settlement_scaled"].to_numpy(np.float32)
            initial_mm = inverse_block([initial_scaled[-1]])[0]
            # Every method keeps its own model-generated state.
            histories = {key: initial_scaled.copy() for key in ("P", "A", "B", "B0")}
            if any(len(sequence) != horizon for sequence in histories.values()):
                raise AssertionError(f"Recursive initialization violates L_k=k at k={horizon}")
            origin_states = {"P": initial_mm, "TA": initial_mm}
            for block_index in range(12 // horizon):
                origin = initial + pd.DateOffset(months=block_index * horizon)
                target = origin + pd.DateOffset(months=horizon)
                if target not in group.index:
                    raise ValueError(f"Missing recursive test endpoint {poi_id}: {target}")
                observed = float(group.loc[target, target_col])  # Scoring only.
                geo = geotechnical_input(poi_id, origin, horizon)
                static = static_lookup[poi_id]
                outputs = {
                    "P": np.full(horizon, origin_states["P"], dtype=float),
                    "T": theoretical_block(poi_id, origin, horizon, ratio),
                    "TA": anchored_theoretical_block(
                        poi_id, origin, origin_states["TA"], horizon, ratio
                    ),
                    "G": inverse_block(predict_model_block(
                        "G", models["G"], initial_scaled, static, geo
                    )),
                }
                for key in ("A", "B", "B0"):
                    outputs[key] = inverse_block(predict_model_block(
                        key, models[key], histories[key], static, geo
                    ))
                for key, block in outputs.items():
                    rows.append({
                        "Protocol": protocol, "Horizon_months": horizon,
                        "Simulation": simulation,
                        "Seed": seed if key in ("A", "B", "B0", "G") else np.nan,
                        "Model_key": key, "poi_id": poi_id,
                        "origin_date": origin, "target_date": target,
                        "observed_settlement_mm": observed,
                        "predicted_settlement_mm": float(block[-1]),
                        "C_alpha_over_Cc": ratio if key in ("T", "TA") else np.nan,
                    })
                    if key in histories:
                        histories[key] = settlement_scaler.transform(
                            np.asarray(block).reshape(-1, 1)
                        ).ravel().astype(np.float32)
                        if len(histories[key]) != horizon:
                            raise AssertionError(f"Recursive input length changed: k={horizon}")
                    if key in origin_states:
                        origin_states[key] = float(block[-1])
    return rows


model_labels = {
    "P": "Persistence", "T": "Unanchored theoretical geotechnical settlement",
    "TA": "Observed-origin-anchored theoretical geotechnical settlement",
    "G": "Geotechnical feature MLP", "A": "InSAR-only LSTM",
    "B": "GeoFusion", "B0": "GeoFusion without geotechnical inputs",
}
prediction_rows = []
draws = np.random.default_rng(base_random_state).uniform(
    *ca_ratio_bounds, size=n_simulations
)
for simulation in range(1, n_simulations + 1):
    seed, ratio = base_random_state + simulation - 1, float(draws[simulation - 1])
    print(f"Run {simulation}/{n_simulations}, C_alpha/Cc={ratio:.5f}")
    for horizon in forecast_horizons:
        ds = datasets[horizon]
        pa = best_params_registry["A"][horizon]
        pb = best_params_registry["B"][horizon]
        pg = best_params_registry["G"][horizon]
        a, _ = train_model(ds, seed, pa, "insar")
        b, _ = train_model(ds, seed, pb, "fusion", pretrained_insar=a)
        # Exact same tuned parameters, transferred A weights and seed as B.
        # The ONLY changed input is the zeroed geotechnical vector.
        b0, _ = train_model(ds, seed, pb, "fusion",
                            pretrained_insar=a, ablate_static=True)
        g, _ = train_geotechnical_model(ds, seed, pg)
        models = {"A": a, "B": b, "B0": b0, "G": g}
        for protocol in ("rolling", "recursive"):
            prediction_rows.extend(make_forecast_rows(
                protocol, horizon, simulation, seed, models, ratio
            ))

predictions = pd.DataFrame(prediction_rows)
predictions["Model"] = predictions["Model_key"].map(model_labels)
predictions.to_csv(output_dir / "all_seven_model_predictions.csv", index=False)


In [ ]:
# RMSE alone, calculated once per run across common borehole--endpoint pairs.
# On shared endpoints, unanchored theoretical predictions must be identical
# across the rolling and recursive protocols for the same ratio draw.
theory_roll = predictions.loc[
    (predictions["Model_key"] == "T") & (predictions["Protocol"] == "rolling")
]
theory_rec = predictions.loc[
    (predictions["Model_key"] == "T") & (predictions["Protocol"] == "recursive")
]
theory_matched = theory_rec.merge(
    theory_roll, on=["Horizon_months", "Simulation", "poi_id", "target_date"],
    suffixes=("_recursive", "_rolling"), validate="one_to_one"
)
if len(theory_matched) != len(theory_rec) or not np.allclose(
    theory_matched["predicted_settlement_mm_recursive"],
    theory_matched["predicted_settlement_mm_rolling"], rtol=0, atol=1e-8
):
    raise AssertionError("Unanchored theoretical rolling/recursive values disagree")
metrics_rows = []
for (protocol, horizon, key, simulation), group in predictions.groupby(
    ["Protocol", "Horizon_months", "Model_key", "Simulation"], sort=True
):
    score = overall_metrics(
        group["observed_settlement_mm"], group["predicted_settlement_mm"]
    )
    metrics_rows.append({
        "Protocol": protocol, "Horizon_months": horizon,
        "Model_key": key, "Model": model_labels[key],
        "Simulation": simulation, "N_predictions": score["N"],
        "RMSE_mm": score["RMSE_mm"],
    })
metrics_by_run = pd.DataFrame(metrics_rows)
model_keys = ("P", "T", "TA", "G", "A", "B", "B0")
expected_runs = set(range(1, n_simulations + 1))
for (protocol, horizon, simulation), block in predictions.groupby(
    ["Protocol", "Horizon_months", "Simulation"]
):
    if set(block["Model_key"]) != set(model_keys):
        raise AssertionError(f"Missing model: {protocol}, k={horizon}, run={simulation}")
    pairs = [set(zip(g["poi_id"], g["target_date"]))
             for _, g in block.groupby("Model_key")]
    if any(pair != pairs[0] for pair in pairs):
        raise AssertionError(f"Unequal held-out targets: {protocol}, k={horizon}, run={simulation}")
for (protocol, horizon, key), block in metrics_by_run.groupby(
    ["Protocol", "Horizon_months", "Model_key"]
):
    if set(block["Simulation"]) != expected_runs:
        raise AssertionError(f"Expected {n_simulations} runs: {protocol}, k={horizon}, {key}")

comparison = metrics_by_run.groupby(
    ["Protocol", "Horizon_months", "Model_key", "Model"], as_index=False
).agg(N_predictions=("N_predictions", "first"),
      N_runs=("Simulation", "nunique"),
      RMSE_mean_mm=("RMSE_mm", "mean"),
      RMSE_SD_mm=("RMSE_mm", lambda v: v.std(ddof=1)))
comparison["RMSE_mean_plusminus_SD_mm"] = comparison.apply(
    lambda row: f'{row["RMSE_mean_mm"]:.2f} ± {row["RMSE_SD_mm"]:.2f}', axis=1
)
rmse_wide = comparison.pivot(
    index=["Protocol", "Horizon_months", "N_predictions"],
    columns="Model_key", values="RMSE_mean_plusminus_SD_mm"
).reset_index().rename(columns={
    "P": "Persistence", "T": "Unanchored theory",
    "TA": "Anchored theory", "G": "Geotechnical MLP",
    "A": "InSAR LSTM", "B": "GeoFusion", "B0": "GeoFusion zero geotech",
})
rmse_wide = rmse_wide[[
    "Protocol", "Horizon_months", "N_predictions", "Persistence",
    "Unanchored theory", "Anchored theory", "Geotechnical MLP", "InSAR LSTM",
    "GeoFusion", "GeoFusion zero geotech",
]]
metrics_by_run.to_csv(output_dir / "seven_model_RMSE_by_run.csv", index=False)
comparison.to_csv(output_dir / "seven_model_RMSE_mean_SD.csv", index=False)
rmse_wide.to_csv(output_dir / "seven_model_RMSE_wide.csv", index=False)
with open(output_dir / "benchmark_configuration.json", "w") as f:
    json.dump({
        "forecast_horizons_months": forecast_horizons,
        "input_sequence_length_months": {str(k): k for k in forecast_horizons},
        "n_simulations": n_simulations,
        "bootstrap_borehole_replicates": n_borehole_bootstraps,
        "theory_T": "unanchored; relative to 2017-01-01; no observed input",
        "theory_TA": "rolling: observed origin; recursive: observed 2020-12 then theoretical increments",
        "theory_Calpha_over_Cc": "global Uniform(0.03, 0.05) per simulation",
        "geotechnical_ablation": "zero STATIC vectors at fit, val and test; B hyperparameters and seed reused",
        "train_fit_boreholes": fit_bhs,
        "train_validation_boreholes": val_bhs,
        "test_boreholes": test_bhs,
        "selected_geotechnical_features": selected_geo,
        "rolling_pairs_by_horizon": {str(k): len(datasets[k]["meta_test"])
                                     for k in forecast_horizons},
    }, f, indent=2)
print(f"RMSE (mm), mean ± sample SD across {n_simulations} runs:")
display(rmse_wide)


In [ ]:
# Paired run-level contrasts. Positive delta means GeoFusion (B) has lower RMSE.
# Simulation IDs use the same A pretraining, tuned parameters and seeds for
# B and B0. Against T or TA, a matched run ID also contains a ratio draw
# independent of the model-training seed; that pairing does not control
# training randomness. T and TA do share their ratio within each run.
comparators = {"B0": "Matched zero-geotech fusion", "A": "InSAR-only LSTM",
               "P": "Persistence", "T": "Unanchored theory",
               "TA": "Observed-origin-anchored theory", "G": "Geotechnical MLP"}
idx_columns = ["Protocol", "Horizon_months", "Simulation"]
wide_runs = metrics_by_run.pivot(
    index=idx_columns, columns="Model_key", values="RMSE_mm"
)
if wide_runs.isna().any().any():
    raise AssertionError("A run or model is missing from the paired RMSE table")
paired_rows = []
for other, description in comparators.items():
    block = wide_runs[["B", other]].copy()
    block["GeoFusion_minus_comparator_RMSE_mm"] = block["B"] - block[other]
    block["RMSE_reduction_by_GeoFusion_mm"] = block[other] - block["B"]
    for (protocol, horizon, simulation), row in block.iterrows():
        paired_rows.append({
            "Protocol": protocol, "Horizon_months": horizon,
            "Simulation": simulation, "Comparator_key": other,
            "Comparator": description,
            "GeoFusion_RMSE_mm": row["B"], "Comparator_RMSE_mm": row[other],
            "RMSE_reduction_by_GeoFusion_mm": row["RMSE_reduction_by_GeoFusion_mm"],
        })
paired_differences = pd.DataFrame(paired_rows)
paired_summary = paired_differences.groupby(
    ["Protocol", "Horizon_months", "Comparator_key", "Comparator"], as_index=False
).agg(
    N_runs=("Simulation", "nunique"),
    Mean_paired_RMSE_reduction_mm=("RMSE_reduction_by_GeoFusion_mm", "mean"),
    SD_paired_RMSE_reduction_mm=("RMSE_reduction_by_GeoFusion_mm",
                                 lambda v: v.std(ddof=1)),
    Fraction_runs_GeoFusion_better=("RMSE_reduction_by_GeoFusion_mm",
                                    lambda v: (v > 0).mean()),
)
paired_differences.to_csv(output_dir / "paired_run_RMSE_differences.csv", index=False)
paired_summary.to_csv(output_dir / "paired_run_RMSE_summary.csv", index=False)
display(paired_summary.loc[paired_summary["Comparator_key"].isin(["B0", "A", "P", "TA"])])

# Direct theory comparison: same layers, ratio draw and endpoints for T and TA.
# Positive reduction indicates that observation anchoring improves RMSE.
theory_effect_by_run = wide_runs[["T", "TA"]].reset_index().rename(columns={
    "T": "Unanchored_theory_RMSE_mm", "TA": "Anchored_theory_RMSE_mm"
})
theory_effect_by_run["RMSE_reduction_by_anchoring_mm"] = (
    theory_effect_by_run["Unanchored_theory_RMSE_mm"]
    - theory_effect_by_run["Anchored_theory_RMSE_mm"]
)
theory_effect_summary = theory_effect_by_run.groupby(
    ["Protocol", "Horizon_months"], as_index=False
).agg(N_runs=("Simulation", "nunique"),
      Mean_RMSE_reduction_by_anchoring_mm=("RMSE_reduction_by_anchoring_mm", "mean"),
      SD_RMSE_reduction_by_anchoring_mm=("RMSE_reduction_by_anchoring_mm",
                                       lambda v: v.std(ddof=1)),
      Fraction_runs_anchoring_better=("RMSE_reduction_by_anchoring_mm",
                                      lambda v: (v > 0).mean()))
theory_effect_by_run.to_csv(output_dir / "theory_anchoring_paired_RMSE_by_run.csv", index=False)
theory_effect_summary.to_csv(output_dir / "theory_anchoring_paired_RMSE_summary.csv", index=False)
display(theory_effect_summary)


In [ ]:
# Borehole-cluster bootstrap of MEAN paired RMSE reduction, not the seed SD.
# Draw borehole IDs with replacement; keep all months for each drawn borehole
# and calculate both model RMSEs on those identical weighted endpoints. The
# same bootstrap weights are used across the 100 seeds. This conditions on the
# trained models and one fit/test split and assumes exchangeable test boreholes.

def borehole_cluster_bootstrap(data, protocol, horizon, comparator,
                               n_bootstraps=2000, random_state=42, focus="B"):
    """Positive RMSE reduction means `focus` beats `comparator`."""
    subset = data.loc[
        (data["Protocol"] == protocol)
        & (data["Horizon_months"] == horizon)
        & data["Model_key"].isin([focus, comparator])
    ].copy()
    subset["squared_error"] = (
        subset["observed_settlement_mm"]
        - subset["predicted_settlement_mm"]
    ) ** 2
    stats = subset.groupby(["Model_key", "Simulation", "poi_id"]).agg(
        sum_squared_error=("squared_error", "sum"), count=("squared_error", "size")
    )
    ids = list(dict.fromkeys(str(pid) for pid in test_bhs))
    run_bh = pd.MultiIndex.from_product(
        [range(1, n_simulations + 1), ids], names=["Simulation", "poi_id"]
    )
    def matrix(key, col):
        series = stats.xs(key, level="Model_key")[col].reindex(run_bh)
        if series.isna().any():
            raise AssertionError(f"Missing {key}/{col} for {protocol}, k={horizon}")
        return series.to_numpy(float).reshape(n_simulations, len(ids))
    sum_b, sum_other = matrix(focus, "sum_squared_error"), matrix(comparator, "sum_squared_error")
    count_b, count_other = matrix(focus, "count"), matrix(comparator, "count")
    if not np.array_equal(count_b, count_other):
        raise AssertionError("Unequal focus/control borehole--endpoint counts")
    if not np.all(count_b == count_b[0]):
        raise AssertionError("Test endpoint count changed across simulation seeds")
    rng = np.random.default_rng(random_state)
    weights = rng.multinomial(len(ids), np.full(len(ids), 1 / len(ids)),
                              size=n_bootstraps).T
    denominator = count_b[0] @ weights
    reduction = (
        np.sqrt((sum_other @ weights) / denominator[None, :])
        - np.sqrt((sum_b @ weights) / denominator[None, :])
    ).mean(axis=0)
    observed = (np.sqrt(sum_other.sum(axis=1) / count_other.sum(axis=1))
                - np.sqrt(sum_b.sum(axis=1) / count_b.sum(axis=1))).mean()
    low, high = np.quantile(reduction, [0.025, 0.975])
    return {"Protocol": protocol, "Horizon_months": horizon,
            "Focus_key": focus, "Comparator_key": comparator, "N_test_boreholes": len(ids),
            "N_predictions_per_run": int(count_b[0].sum()),
            "Mean_paired_RMSE_reduction_mm": float(observed),
            "Borehole_bootstrap_2p5_percent_mm": float(low),
            "Borehole_bootstrap_97p5_percent_mm": float(high),
            "N_borehole_bootstraps": n_bootstraps}

bootstrap_rows = []
for protocol in ("rolling", "recursive"):
    for horizon in forecast_horizons:
        for other in ("B0", "A", "P"):
            bootstrap_rows.append(borehole_cluster_bootstrap(
                predictions, protocol, horizon, other,
                n_bootstraps=n_borehole_bootstraps,
                random_state=base_random_state + horizon + (0 if protocol == "rolling" else 1000),
            ))
        bootstrap_rows.append(borehole_cluster_bootstrap(
            predictions, protocol, horizon, "T", focus="TA",
            n_bootstraps=n_borehole_bootstraps,
            random_state=base_random_state + horizon + (0 if protocol == "rolling" else 1000),
        ))
bootstrap_summary = pd.DataFrame(bootstrap_rows)
bootstrap_summary.to_csv(output_dir / "paired_borehole_bootstrap_RMSE.csv", index=False)
display(bootstrap_summary)


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
try:
    from IPython.display import display
except ImportError:
    def display(frame):
        print(frame.to_string(index=False))

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.labelsize': 10,
    'axes.titlesize': 11,
    'figure.dpi': 110,
    'savefig.dpi': 220,
    'axes.spines.top': False,
    'axes.spines.right': False,
})


In [ ]:
# Adjust these paths for the working directory where you open the notebook.
BENCHMARK_OUTPUT_DIR = Path('seven_method_geofusion_ablation_100_runs')
PREDICTION_CSV = BENCHMARK_OUTPUT_DIR / 'all_seven_model_predictions.csv'
INSAR_CSV = Path(
    '../../../Japan_Airport/script/vertical_displacement_csv_NewJuly2/'
    'settlement_selected_POIs_positive_downward.csv'
)
TARGET_COL = 'settlement_mm_positive_downward'
PLOT_OUTPUT_DIR = BENCHMARK_OUTPUT_DIR / 'four_method_six_borehole_forecast_plots'

BOREHOLES = ('478', '555', '630', '886', '1047', '4803')
HORIZONS = (1, 3, 6, 12)
PROTOCOLS = ('recursive',)  # Optional: ('recursive', 'rolling')
EXPECTED_RUNS = 100
SHOW_RUN_SD = False  # Optional: run-to-run variation, NOT new-borehole uncertainty.

MODEL_LABELS = {
    'G': 'Geotechnical MLP',
    'TA': 'Anchored theory',
    'A': 'InSAR LSTM',
    'B': 'GeoFusion',
}
MODEL_COLORS = {'G': '#9467bd', 'TA': '#c56a11', 'A': '#277da1', 'B': '#18855b'}
MODEL_MARKERS = {'G': 's', 'TA': '^', 'A': 'o', 'B': 'D'}

for path in (PREDICTION_CSV, INSAR_CSV):
    if not path.is_file():
        raise FileNotFoundError(f'Update the Settings cell: file not found: {path}')
PLOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Recreate exactly the benchmark's monthly mean InSAR history.
insar = pd.read_csv(INSAR_CSV)
required_insar = {'poi_id', 'date', TARGET_COL}
missing = required_insar.difference(insar.columns)
if missing:
    raise ValueError(f'InSAR CSV missing columns: {sorted(missing)}')
insar['poi_id'] = insar['poi_id'].astype(str).str.strip()
insar['date'] = pd.to_datetime(insar['date'], errors='coerce')
insar[TARGET_COL] = pd.to_numeric(insar[TARGET_COL], errors='coerce')
insar = insar.dropna(subset=['poi_id', 'date', TARGET_COL])
insar = insar[
    insar['poi_id'].isin(BOREHOLES)
    & insar['date'].between('2017-01-01', '2020-12-31')
].copy()
if insar.empty:
    raise ValueError('No 2017--2020 InSAR observations found for the selected boreholes.')
observed = pd.concat([
    group.sort_values('date').set_index('date')[[TARGET_COL]]
         .resample('MS').mean().dropna().assign(poi_id=bh).reset_index()
    for bh, group in insar.groupby('poi_id')
], ignore_index=True).rename(columns={'date': 'month', TARGET_COL: 'observed_mm'})
missing_history = set(BOREHOLES) - set(observed['poi_id'])
if missing_history:
    raise ValueError(f'No historical observations for boreholes: {sorted(missing_history)}')
for bh in BOREHOLES:
    bh_history = observed.loc[observed['poi_id'] == bh]
    if pd.Timestamp('2020-12-01') not in set(bh_history['month']):
        raise ValueError(f'Missing December 2020 observed forecast origin at borehole {bh}')

# Read only the four selected methods; do not plot the saved 2021 observed targets.
pred = pd.read_csv(PREDICTION_CSV, dtype={'poi_id': str, 'Model_key': str})
required_pred = {
    'Protocol', 'Horizon_months', 'Simulation', 'Model_key', 'poi_id',
    'target_date', 'predicted_settlement_mm',
}
missing = required_pred.difference(pred.columns)
if missing:
    raise ValueError(f'Prediction CSV missing columns: {sorted(missing)}')
pred['poi_id'] = pred['poi_id'].str.strip()
pred['target_date'] = pd.to_datetime(pred['target_date'], errors='coerce')
pred['predicted_settlement_mm'] = pd.to_numeric(
    pred['predicted_settlement_mm'], errors='coerce'
)
pred = pred[
    pred['poi_id'].isin(BOREHOLES)
    & pred['Protocol'].isin(PROTOCOLS)
    & pred['Horizon_months'].isin(HORIZONS)
    & pred['Model_key'].isin(MODEL_LABELS)
].copy()
if pred.empty or pred[['target_date', 'predicted_settlement_mm', 'Simulation']].isna().any().any():
    raise ValueError('Missing or invalid saved forecasts for the selected settings.')
if not np.isfinite(pred['predicted_settlement_mm'].to_numpy()).all():
    raise ValueError('Saved forecast values must be finite.')
if not pred['target_date'].between('2021-01-01', '2021-12-01').all():
    raise ValueError('Expected 2021 forecast endpoints only.')

# Ensure every panel compares four approaches at exactly the same dates and runs.
unique_key = ['Protocol', 'Horizon_months', 'poi_id', 'Model_key', 'Simulation', 'target_date']
if pred.duplicated(unique_key).any():
    raise ValueError('The prediction CSV has duplicate model/run/endpoint rows.')
for protocol in PROTOCOLS:
    for horizon in HORIZONS:
        expected_dates = (
            {pd.Timestamp('2020-12-01') + pd.DateOffset(months=j*horizon)
             for j in range(1, 12//horizon + 1)}
            if protocol == 'recursive'
            else set(pd.date_range('2021-01-01', '2021-12-01', freq='MS'))
        )
        for bh in BOREHOLES:
            sub = pred[(pred['Protocol'] == protocol)
                       & (pred['Horizon_months'] == horizon)
                       & (pred['poi_id'] == bh)]
            if set(sub['Model_key']) != set(MODEL_LABELS):
                raise ValueError(f'Missing models for {protocol}, k={horizon}, borehole {bh}')
            for key in MODEL_LABELS:
                rows = sub[sub['Model_key'] == key]
                counts = rows.groupby('target_date')['Simulation'].nunique()
                if set(counts.index) != expected_dates or not (counts == EXPECTED_RUNS).all():
                    raise ValueError(
                        f'Expected {EXPECTED_RUNS} runs at all scored {protocol} '
                        f'k={horizon} endpoints for {bh}/{key}; got {counts.to_dict()}'
                    )
                if rows['Simulation'].nunique() != EXPECTED_RUNS:
                    raise ValueError(f'Inconsistent run identifiers for {protocol}, k={horizon}, {bh}/{key}')

summary = pred.groupby(
    ['Protocol', 'Horizon_months', 'poi_id', 'Model_key', 'target_date'],
    as_index=False,
).agg(
    mean_mm=('predicted_settlement_mm', 'mean'),
    sd_runs_mm=('predicted_settlement_mm', lambda x: x.std(ddof=1)),
    n_runs=('Simulation', 'nunique'),
)
print(f'Validated {len(BOREHOLES)} boreholes, {len(PROTOCOLS)} protocol(s), '
      f'{len(HORIZONS)} horizons and {EXPECTED_RUNS} runs per model/endpoint.')
display(summary.head(8))


## Interpreting the ablation and uncertainty

- B vs B0 is the central geotechnical-feature test. Both share the GeoFusion
  architecture, pretrained InSAR branch, tuned hyperparameters and seed; only
  availability of the *mean geotechnical input values* changes. It is an
  input ablation, not a causal identification of soil processes. Hyperparameters
  tuned for B are reused for B0, so the comparison is conditional on that
  shared configuration rather than separately optimized architectures.
- B vs A tests the practical added value of the fusion implementation, but
  its architecture and number of trainable parameters also change.
- A positive `RMSE_reduction_by_GeoFusion_mm` means lower GeoFusion RMSE;
  negative values mean GeoFusion is worse. Interpret comparisons only within
  each protocol and horizon. Rolling and recursive endpoints and inputs differ.
- `SD_paired_RMSE_reduction_mm` describes run-to-run training variation on
  the same boreholes. It is not uncertainty for previously unseen boreholes.
- Borehole bootstrap limits are descriptive sensitivity intervals from the
  retained test boreholes (at most 17). Neighboring boreholes may share deformation conditions,
  so the resampling independence assumption can fail. No formal hypothesis
  test or strong generalization claim follows from these bounds alone.
- T is the unanchored 2017-referenced theoretical benchmark. TA uses the
  same layered law and the same C-alpha/Cc draw but adds an InSAR-observed
  settlement anchor at each rolling origin, or at December 2020 for recursive
  forecasts. Thus T vs TA measures the value of additional origin information;
  it does not isolate a change to the geotechnical material model. A positive
  `RMSE_reduction_by_anchoring_mm` indicates lower RMSE for TA.
- G uses static geotechnical and known calendar features only. Its training
  targets are observed pre-2021 InSAR settlements, but its forecast inputs
  contain no observed settlement history. T and TA standard deviations
  reflect ratio draws; G standard deviations reflect model training.
- Refit and evaluate a second independent spatial test cohort, if available,
  before claiming a general site-wide advantage for geotechnical features.
- The source CSVs are required to execute this notebook. No results are
  fabricated or carried over from the earlier benchmark.
